In [3]:
pip install kagglehub

Note: you may need to restart the kernel to use updated packages.


## Pregunta Inicial
- **Hipótesis en Bruto**:
	- Quien tiene mayor porcentaje de victorias Strikers o Grapplers?
	- Quien tiene peleas mas largas Strikers o Grapplers?
	- Cual es el porcentaje de victorias por tipo de peleador? (Strikers Vs Grapplers)
	- Que marca la diferencia en los encuentros?
	- Con que atributos esta relacionada la apuesta del peleador?
- **Hipótesis Finales (Base)**:
	- Existe una relación ente las apuestas (Favorito Vs UnderDog) con la victoria?
	- De qué manera esta relacionado el arquetipo del peleador con el resultado de la pelea? (Tiempo, Manera de Finalización)
	- Existe un patrón que explique el fenómeno de "La Invasión Daguestani"?

### Descargando Dataset 

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mdabbert/ultimate-ufc-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/scoria/.cache/kagglehub/datasets/mdabbert/ultimate-ufc-dataset/versions/181


### Importando Bibliotecas

In [5]:
import pandas as pd

### Cargando el DataFrame

In [6]:
df_ufc = pd.read_csv(f"{path}/ufc-master.csv")

*Primera Vista*

In [7]:
df_ufc.head()

,R_fighter,B_fighter,R_odds,B_odds,R_ev,B_ev,date,location,country,Winner,...,finish_details,finish_round,finish_round_time,total_fight_time_secs,r_dec_odds,b_dec_odds,r_sub_odds,b_sub_odds,r_ko_odds,b_ko_odds
0,Israel Adesanya,Joe Pyfer,-130.0,102.0,76.9231,102.0000,2026-03-28,"Seattle, Washington, USA",USA,Blue,...,Punches,2.0,4:18,558.0,163.0,900.0,2500.0,400.0,300.0,250.0
1,Alexa Grasso,Maycee Barber,124.0,-158.0,124.0000,63.2911,2026-03-28,"Seattle, Washington, USA",USA,Red,...,Punch,1.0,2:42,162.0,175.0,105.0,1400.0,800.0,2500.0,500.0
2,Michael Chiesa,Niko Price,-901.0,550.0,11.0988,550.0000,2026-03-28,"Seattle, Washington, USA",USA,Red,...,Rear Naked Choke,1.0,1:03,63.0,225.0,900.0,-150.0,1600.0,600.0,1000.0
3,Julian Erosa,Lerryan Douglas,235.0,-320.0,235.0000,31.2500,2026-03-28,"Seattle, Washington, USA",USA,Blue,...,Punches,1.0,3:33,213.0,600.0,500.0,600.0,2000.0,700.0,-150.0
4,Mansur Abdul-Malik,Yousri Belgaroui,-158.0,124.0,63.2911,124.0000,2026-03-28,"Seattle, Washington, USA",USA,Blue,...,Knee,3.0,3:39,819.0,350.0,240.0,800.0,1800.0,240.0,250.0


## 1. Sanity Check

In [8]:
print(df_ufc.shape)

(7177, 118)


In [9]:
df_ufc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7177 entries, 0 to 7176
Columns: 118 entries, R_fighter to b_ko_odds
dtypes: bool(1), float64(60), int64(43), object(14)
memory usage: 6.4+ MB


In [10]:
df_ufc.columns

Index(['R_fighter', 'B_fighter', 'R_odds', 'B_odds', 'R_ev', 'B_ev', 'date',
       'location', 'country', 'Winner',
       ...
       'finish_details', 'finish_round', 'finish_round_time',
       'total_fight_time_secs', 'r_dec_odds', 'b_dec_odds', 'r_sub_odds',
       'b_sub_odds', 'r_ko_odds', 'b_ko_odds'],
      dtype='object', length=118)

In [11]:
df_ufc['R_fighter'].unique()

array(['Israel Adesanya', 'Alexa Grasso', 'Michael Chiesa', ...,
       'Caol Uno', 'Eliot Marshall', 'Eric Schafer'],
      shape=(1771,), dtype=object)

In [12]:
df_ufc['B_fighter'].unique()

array(['Joe Pyfer', 'Maycee Barber', 'Niko Price', ..., 'James Irvin',
       'Shannon Gugerty', 'Chase Gormley'], shape=(2054,), dtype=object)

In [42]:
df_ufc.describe()

,R_odds,B_odds,R_ev,B_ev,no_of_rounds,B_current_lose_streak,B_current_win_streak,B_draw,B_avg_SIG_STR_landed,B_avg_SIG_STR_pct,...,B_Flyweight_rank,B_Pound-for-Pound_rank,finish_round,total_fight_time_secs,r_dec_odds,b_dec_odds,r_sub_odds,b_sub_odds,r_ko_odds,b_ko_odds
count,6937.000000,6938.000000,6937.000000,6938.000000,7177.000000,7177.000000,7177.000000,7177.000000,6247.000000,6412.000000,...,143.000000,81.000000,6555.000000,6555.000000,6062.000000,6032.000000,5813.000000,5789.000000,5815.000000,5788.000000
mean,-114.380280,56.003459,98.775801,165.136205,3.187126,0.502996,1.002229,0.025916,18.211118,0.456023,...,8.440559,9.370370,2.423036,657.684821,315.757176,431.808687,995.132118,1211.647608,596.026827,729.980477
std,285.370533,260.799641,90.112714,140.153769,0.579962,0.797343,1.465126,0.164074,19.825099,0.108546,...,4.284960,4.357305,1.009258,360.670777,260.648779,337.468752,816.926221,903.365182,638.926484,699.021015
min,-2100.000000,-2200.000000,4.761900,4.545500,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,5.000000,-440.000000,-200.000000,-370.000000,-1250.000000,-900.000000,-400.000000
25%,-255.000000,-150.000000,39.215700,66.666700,3.000000,0.000000,0.000000,0.000000,3.745000,0.400000,...,5.000000,5.000000,1.000000,299.000000,170.000000,220.000000,450.000000,600.000000,240.000000,320.000000
50%,-148.000000,130.000000,67.567600,130.000000,3.000000,0.000000,0.000000,0.000000,6.120000,0.460000,...,8.000000,10.000000,3.000000,900.000000,260.000000,350.000000,800.000000,1000.000000,450.000000,550.000000
75%,130.000000,215.000000,130.000000,215.000000,3.000000,1.000000,1.000000,0.000000,30.433050,0.520000,...,12.000000,14.000000,3.000000,900.000000,420.000000,550.000000,1300.000000,1550.000000,762.500000,900.000000
max,1100.000000,1300.000000,1100.000000,1300.000000,5.000000,6.000000,15.000000,2.000000,154.000000,1.000000,...,15.000000,15.000000,5.000000,1500.000000,2400.000000,3500.000000,10000.000000,12500.000000,5000.000000,6600.000000


## Atributos de Acciones

In [49]:
df_ufc[['B_avg_SIG_STR_landed', 'R_avg_SIG_STR_landed', 'B_avg_SIG_STR_pct', 'R_avg_SIG_STR_pct', 'B_avg_SUB_ATT', 'R_avg_SUB_ATT', 'B_avg_TD_landed', 'R_avg_TD_landed', 'B_avg_TD_pct', 'R_avg_TD_pct']].describe()

,B_avg_SIG_STR_landed,R_avg_SIG_STR_landed,B_avg_SIG_STR_pct,R_avg_SIG_STR_pct,B_avg_SUB_ATT,R_avg_SUB_ATT,B_avg_TD_landed,R_avg_TD_landed,B_avg_TD_pct,R_avg_TD_pct
count,6247.000000,6722.000000,6412.000000,6820.000000,6345.000000,6820.000000,6344.000000,6820.000000,6335.000000,6810.000000
mean,18.211118,19.529597,0.456023,0.462621,0.499645,0.535286,1.347529,1.428983,0.326992,0.344957
std,19.825099,19.547405,0.108546,0.096235,0.678532,0.703734,1.377428,1.344243,0.236926,0.219053
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.745000,3.990000,0.400000,0.410000,0.000000,0.000000,0.333300,0.461500,0.160000,0.206000
50%,6.120000,8.170000,0.460000,0.460000,0.300000,0.333300,1.000000,1.070000,0.330000,0.340000
75%,30.433050,32.000000,0.520000,0.520000,0.800000,0.800000,2.000000,2.000000,0.470000,0.472000
max,154.000000,141.000000,1.000000,1.000000,8.400000,14.300000,10.860000,14.290000,1.000000,1.000000


In [14]:
cols_interes = [
    'fighter', 'avg_SIG_STR_landed', 'avg_SIG_STR_pct', 
    'avg_SUB_ATT', 'avg_TD_landed', 'avg_TD_pct', 
    'Height_cms', 'Reach_cms', 'Weight_lbs', 'age', 'Stance'
]

In [15]:
df_red = df_ufc[['R_' + col for col in cols_interes]].copy()
df_red.columns = cols_interes

In [16]:
df_blue = df_ufc[['B_' + col for col in cols_interes]].copy()
df_blue.columns = cols_interes

In [17]:
df_red.head()

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,Height_cms,Reach_cms,Weight_lbs,age,Stance
0,Israel Adesanya,4.03,0.48,0.1,0.05,0.09,193.04,203.20,185,36,Switch
1,Alexa Grasso,4.11,0.41,0.7,0.40,0.35,165.10,167.64,125,32,Orthodox
2,Michael Chiesa,2.02,0.40,1.0,3.11,0.47,185.42,190.50,170,38,Southpaw
3,Julian Erosa,6.18,0.48,0.7,1.73,0.43,185.42,187.96,145,36,Southpaw
4,Mansur Abdul-Malik,3.28,0.44,0.3,1.65,0.41,187.96,203.20,185,28,Orthodox


In [18]:
df_red.tail()

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,Height_cms,Reach_cms,Weight_lbs,age,Stance
7172,Duane Ludwig,13.6667,0.577,0.0000,0.0000,0.000,177.80,177.80,170,31,Orthodox
7173,John Howard,18.0000,0.550,1.0000,4.6667,0.790,170.18,180.34,170,27,Orthodox
7174,Brendan Schaub,12.0000,0.250,0.0000,0.0000,0.000,193.04,198.12,245,27,Orthodox
7175,Mike Pierce,40.5000,0.405,0.0000,3.5000,0.520,172.72,177.80,170,29,Orthodox
7176,Eric Schafer,15.6667,0.588,1.3333,0.8333,0.145,190.50,190.50,185,32,Orthodox


In [19]:
df_red.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7177 entries, 0 to 7176
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   fighter             7177 non-null   object 
 1   avg_SIG_STR_landed  6722 non-null   float64
 2   avg_SIG_STR_pct     6820 non-null   float64
 3   avg_SUB_ATT         6820 non-null   float64
 4   avg_TD_landed       6820 non-null   float64
 5   avg_TD_pct          6810 non-null   float64
 6   Height_cms          7177 non-null   float64
 7   Reach_cms           7177 non-null   float64
 8   Weight_lbs          7177 non-null   int64  
 9   age                 7177 non-null   int64  
 10  Stance              7177 non-null   object 
dtypes: float64(7), int64(2), object(2)
memory usage: 616.9+ KB


In [20]:
df_blue.head()

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,Height_cms,Reach_cms,Weight_lbs,age,Stance
0,Joe Pyfer,3.52,0.44,0.9,1.45,0.30,187.96,190.50,185,29,Orthodox
1,Maycee Barber,4.56,0.53,0.1,1.56,0.45,165.10,165.10,125,27,Switch
2,Niko Price,5.11,0.43,0.6,1.06,0.30,182.88,193.04,170,36,Orthodox
3,Lerryan Douglas,8.67,0.64,0.0,0.00,0.00,175.26,182.88,145,30,Orthodox
4,Yousri Belgaroui,6.10,0.64,0.0,0.29,1.00,198.12,200.66,185,33,Orthodox


In [21]:
df_blue.tail()

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,Height_cms,Reach_cms,Weight_lbs,age,Stance
7172,Darren Elkins,NaN,NaN,NaN,NaN,NaN,177.80,180.34,145,25,Orthodox
7173,Daniel Roberts,NaN,NaN,NaN,NaN,NaN,177.80,187.96,170,29,Southpaw
7174,Chase Gormley,8.0000,0.34,1.0000,1.0000,1.0,190.50,196.00,265,27,Orthodox
7175,Julio Paulino,NaN,NaN,NaN,NaN,NaN,182.88,185.42,170,34,Orthodox
7176,Jason Brilz,31.6667,0.46,0.6667,1.6667,0.5,180.34,180.34,205,34,Orthodox


In [22]:
df_blue.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7177 entries, 0 to 7176
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   fighter             7177 non-null   object 
 1   avg_SIG_STR_landed  6247 non-null   float64
 2   avg_SIG_STR_pct     6412 non-null   float64
 3   avg_SUB_ATT         6345 non-null   float64
 4   avg_TD_landed       6344 non-null   float64
 5   avg_TD_pct          6335 non-null   float64
 6   Height_cms          7177 non-null   float64
 7   Reach_cms           7177 non-null   float64
 8   Weight_lbs          7177 non-null   int64  
 9   age                 7177 non-null   int64  
 10  Stance              7171 non-null   object 
dtypes: float64(7), int64(2), object(2)
memory usage: 616.9+ KB


In [23]:
df_red_blue = pd.concat([df_red, df_blue], axis=0, ignore_index=True)

In [24]:
df_red_blue.head()

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,Height_cms,Reach_cms,Weight_lbs,age,Stance
0,Israel Adesanya,4.03,0.48,0.1,0.05,0.09,193.04,203.20,185,36,Switch
1,Alexa Grasso,4.11,0.41,0.7,0.40,0.35,165.10,167.64,125,32,Orthodox
2,Michael Chiesa,2.02,0.40,1.0,3.11,0.47,185.42,190.50,170,38,Southpaw
3,Julian Erosa,6.18,0.48,0.7,1.73,0.43,185.42,187.96,145,36,Southpaw
4,Mansur Abdul-Malik,3.28,0.44,0.3,1.65,0.41,187.96,203.20,185,28,Orthodox


In [25]:
df_red_blue.tail()

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,Height_cms,Reach_cms,Weight_lbs,age,Stance
14349,Darren Elkins,NaN,NaN,NaN,NaN,NaN,177.80,180.34,145,25,Orthodox
14350,Daniel Roberts,NaN,NaN,NaN,NaN,NaN,177.80,187.96,170,29,Southpaw
14351,Chase Gormley,8.0000,0.34,1.0000,1.0000,1.0,190.50,196.00,265,27,Orthodox
14352,Julio Paulino,NaN,NaN,NaN,NaN,NaN,182.88,185.42,170,34,Orthodox
14353,Jason Brilz,31.6667,0.46,0.6667,1.6667,0.5,180.34,180.34,205,34,Orthodox


In [26]:
df_red_blue.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14354 entries, 0 to 14353
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   fighter             14354 non-null  object 
 1   avg_SIG_STR_landed  12969 non-null  float64
 2   avg_SIG_STR_pct     13232 non-null  float64
 3   avg_SUB_ATT         13165 non-null  float64
 4   avg_TD_landed       13164 non-null  float64
 5   avg_TD_pct          13145 non-null  float64
 6   Height_cms          14354 non-null  float64
 7   Reach_cms           14354 non-null  float64
 8   Weight_lbs          14354 non-null  int64  
 9   age                 14354 non-null  int64  
 10  Stance              14348 non-null  object 
dtypes: float64(7), int64(2), object(2)
memory usage: 1.2+ MB


In [27]:
df_red_blue['fighter'].unique()

array(['Israel Adesanya', 'Alexa Grasso', 'Michael Chiesa', ...,
       'Paul Buentello', 'Shannon Gugerty', 'Chase Gormley'],
      shape=(2241,), dtype=object)

In [28]:
df_red_blue.dropna

<bound method DataFrame.dropna of                   fighter  avg_SIG_STR_landed  avg_SIG_STR_pct  avg_SUB_ATT  \
0         Israel Adesanya              4.0300             0.48       0.1000   
1            Alexa Grasso              4.1100             0.41       0.7000   
2          Michael Chiesa              2.0200             0.40       1.0000   
3            Julian Erosa              6.1800             0.48       0.7000   
4      Mansur Abdul-Malik              3.2800             0.44       0.3000   
...                   ...                 ...              ...          ...   
14349       Darren Elkins                 NaN              NaN          NaN   
14350      Daniel Roberts                 NaN              NaN          NaN   
14351       Chase Gormley              8.0000             0.34       1.0000   
14352       Julio Paulino                 NaN              NaN          NaN   
14353         Jason Brilz             31.6667             0.46       0.6667   

       avg_TD_lan

In [51]:
df_peleador_acciones = df_red_blue.groupby('fighter').agg({
    'avg_SIG_STR_landed': 'mean',
    'avg_SIG_STR_pct': 'mean',
    'avg_SUB_ATT': 'mean',
    'avg_TD_landed': 'mean',
    'avg_TD_pct': 'mean'
}).reset_index()

In [52]:
df_peleador_caracteristicas = df_red_blue.groupby('fighter').agg({
    'Height_cms': 'last',
    'Reach_cms': 'last',
    'Weight_lbs': 'last',
    'age': 'max',
    'Stance': 'last'
}).reset_index()

In [53]:
df_peleador_acciones = df_peleador_acciones.dropna(subset=['fighter'])

In [54]:
df_peleador_acciones.shape

(2241, 6)

In [55]:
df_peleador_acciones.head()

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct
0,Jun Yong Park,NaN,0.522,0.00,0.50,0.25
1,AJ Cunningham,5.94,0.290,0.25,0.00,0.00
2,AJ Dobson,4.29,0.460,0.30,1.67,0.75
3,AJ Fletcher,3.36,0.490,0.90,1.54,0.35
4,Aalon Cruz,8.31,0.395,0.00,0.00,0.00


In [56]:
df_peleador_acciones.tail()

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct
2236,Zhang Weili,5.577778,0.514444,0.544444,2.162222,0.412222
2237,Zu Anyanwu,NaN,NaN,NaN,NaN,NaN
2238,Zubaira Tukhugov,16.528571,0.392143,0.000000,2.286186,0.450000
2239,Zviad Lazishvili,4.200000,0.390000,0.000000,0.000000,0.000000
2240,Zygimantas Ramaska,0.960000,0.600000,2.400000,0.000000,0.000000


In [57]:
df_peleador_acciones.dropna()

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct
1,AJ Cunningham,5.940000,0.290000,0.250000,0.000000,0.000000
2,AJ Dobson,4.290000,0.460000,0.300000,1.670000,0.750000
3,AJ Fletcher,3.360000,0.490000,0.900000,1.540000,0.350000
4,Aalon Cruz,8.310000,0.395000,0.000000,0.000000,0.000000
5,Aaron Phillips,7.126667,0.523333,0.700000,0.286667,0.166667
...,...,...,...,...,...,...
2235,Zhang Mingyang,8.305000,0.580000,0.000000,0.000000,0.000000
2236,Zhang Weili,5.577778,0.514444,0.544444,2.162222,0.412222
2238,Zubaira Tukhugov,16.528571,0.392143,0.000000,2.286186,0.450000
2239,Zviad Lazishvili,4.200000,0.390000,0.000000,0.000000,0.000000


In [58]:
df_peleador_acciones.describe()

,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct
count,2135.000000,2149.000000,2144.000000,2144.000000,2144.000000
mean,16.952937,0.447141,0.485682,1.317959,0.310135
std,16.403671,0.112304,0.766311,1.423511,0.236860
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.049545,0.387667,0.000000,0.257215,0.125000
50%,10.136108,0.452500,0.228571,0.925248,0.301321
75%,26.831650,0.514444,0.712620,1.966615,0.458500
max,121.000000,0.860000,14.300000,14.290000,1.000000


In [59]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Ilia Topuria']

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct
845,Ilia Topuria,3.415556,0.406667,1.755556,2.223333,0.506667


In [60]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Diego Lopes']

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct
587,Diego Lopes,3.65,0.518889,2.733333,0.551111,0.366667


In [61]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'].str.startswith('Conor'), 'fighter']

455    Conor McGregor
Name: fighter, dtype: object

In [62]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Georges St-Pierre']

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct
758,Georges St-Pierre,53.763657,0.570857,1.1873,3.889357,0.746714


In [63]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Jon Jones']

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct
1061,Jon Jones,35.343458,0.561263,0.516753,2.063526,0.589316


In [64]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Conor McGregor']

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct
455,Conor McGregor,29.551292,0.476923,0.007692,0.870377,0.366308
